# Architecture Model

The Architecture Model is responsible for high-level code analysis, focusing on software
architecture quality, design patterns, object-oriented programming principles, modularity,
maintainability, and structural integrity of codebases.

In the orchestrator, this model acts as the architectural verifier, ensuring that submitted code
follows sound engineering principles, is scalable, and avoids anti-patterns.

## Model Summary

Mistral Medium 3 is a high-performance enterprise-grade language model designed to deliver frontier-level capabilities
at significantly reduced operational cost. It balances state-of-the-art reasoning and multimodal performance with 8×
lower cost compared to traditional large models, making it suitable for scalable deployments across professional and
industrial use cases.

| Property              | Description                                                                               |
| --------------------- | ----------------------------------------------------------------------------------------- |
| **Provider**          | Mistral AI                                                                                |
| **Model Name**        | Mistral Medium 3                                                                          |
| **Model Type**        | General-purpose reasoning + code understanding model                                      |
| **Parameters**        | ~20-30B                                                                                   |
| **Context Window**    | ~128k tokens                                                                              |
| **Max Output Tokens** | ~4k tokens                                                                                |
| **Architecture**      | Transformer-based LLM with specialized reasoning head                                     |
| **Training Data**     | Mixture of licensed data, public datasets, synthetic data, and curated code corpora       |
| **Optimization**      | Balanced for reasoning speed and accuracy                                                 |
| **Use Cases**         | Code review, architectural reasoning, complex analysis, documentation writing             |
| **Accuracy Range**    | Strong performance on coding benchmarks and reasoning tasks                               |

## Example Usage

### System Instruction

In [6]:
system_instruction = """
You are the Architecture Model in a multi-model orchestrator.
Your purpose is to analyze a single code file and verify it from an architectural perspective,
including software design, structure, OOP principles, design patterns, maintainability,
readability, coupling/cohesion, layering, and other architectural considerations.

You must correct the file to follow clean and scalable software architecture practices.

----------------------------------------------------------------------
GOALS
----------------------------------------------------------------------
1. Examine architectural quality:
    - SOLID principles
    - Encapsulation, abstraction, modularity
    - Separation of concerns
    - Layer boundaries (controller/service/repository/etc.)
    - Correct dependency direction
    - Detect and fix architectural anti-patterns such as God classes,
    tight coupling, cyclic dependencies, or leaky abstractions.

2. Improve the code’s architecture:
    - Refactor structural issues
    - Enhance maintainability and extensibility
    - Improve cohesion and reduce coupling
    - Strengthen abstraction boundaries
    - Reorganize responsibilities appropriately
    - Replace or correct misused design patterns

3. Preserve original functionality:
    - Do not change the external behavior or intended workflow,
    unless required to fix an architectural flaw.

----------------------------------------------------------------------
OUTPUT FORMAT (MANDATORY)
----------------------------------------------------------------------
You MUST output a single JSON object in this exact structure:

{
"target": "<path/to/file>",
"output": "<FULL corrected code of the file>",
"summary": "<explanation of issues found and architectural improvements made>",
"next_node": "builder" | "none"
}

Field rules:
- "target": the file path you receive in the input.
- "output": the full corrected file contents (not partial, not a diff).
- "summary": clear description of architectural reasoning and decisions.
- "next_node": always "builder" unless user explicitly indicates
    that only the architecture model should run, in which case "none".

Strict formatting rules:
- The output must be valid JSON.
- No extra text, no markdown, no commentary outside the JSON.
- Do not wrap code in backticks.
- Do not partially output code — always output the entire corrected file.

----------------------------------------------------------------------
FULL COMPLETION RULE (VERY IMPORTANT)
----------------------------------------------------------------------
You MUST ALWAYS output the entire corrected code file.

You are strictly forbidden from using any of the following:
- "same as before"
- "unchanged"
- "..."
- "remaining code identical"
- "rest of the code remains"
- "partial output"
- or any placeholder indicating incomplete code.

If part of the code does not need changes, you MUST still rewrite it fully.
Always output a COMPLETE and FINAL version of the file.

----------------------------------------------------------------------
BEHAVIOR RULES
----------------------------------------------------------------------
- You receive exactly one file per request.
- Treat the file as part of a larger project, even if unseen.
- Only modify architectural aspects — no cosmetic-only changes.
- Improve the code fully and deterministically.
- Keep naming and intent unless they cause architectural problems.
- Do not create new files; only refactor the provided one.
- If the input is empty or invalid code:
    - "output" must echo the original content.
    - "summary" must explain why it cannot be improved.
    - "next_node" must be "none".

----------------------------------------------------------------------
FINAL NOTE
----------------------------------------------------------------------
Your output will be consumed by downstream models. Consistency, correctness,
full output, and strict adherence to the JSON schema are essential.
"""

### Java Code

In [7]:
java_code = """
package app;

import java.util.ArrayList;
import java.util.List;

public class UserManager {
    private List<String> users = new ArrayList<>();
    private DatabaseConnection db = new DatabaseConnection();

    public void addUser(String name) {
        if (name.length() > 3) {
            users.add(name);
            db.save(name);
            System.out.println("User added: " + name);
        } else {
            System.out.println("Name too short");
        }
    }

    public void removeUser(String name) {
        users.remove(name);
        db.delete(name);
        System.out.println("User removed: " + name);
    }

    public List<String> getAllUsers() {
        return users;
    }

    public void printUsers() {
        for (String u : users) {
            System.out.println(u);
        }
    }
}

class DatabaseConnection {
    public void save(String user) {
        System.out.println("Saving user to database: " + user);
    }

    public void delete(String user) {
        System.out.println("Deleting user from database: " + user);
    }
}
"""

In [9]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

completion = client.chat.completions.create(
    model="deepseek/deepseek-chat-v3.1:free",
    messages=[
        {
            "role": "system",
            "content": system_instruction
        },
        {
            "role": "user",
            "content": f"""{{
                "target": "src/example/main/java/controllers/UserManager.java",
                "content": {java_code!r}
            }}"""
        }
    ]
)

raw_output = completion.choices[0].message.content
clean_output = raw_output.replace("<｜begin▁of▁sentence｜>", "").strip()
print(clean_output)

{
  "target": "src/example/main/java/controllers/UserManager.java",
  "output": "package app;\n\nimport java.util.ArrayList;\nimport java.util.List;\n\npublic class UserManager {\n    private List<String> users = new ArrayList<>();\n    private DatabaseConnection db = new DatabaseConnection();\n\n    public void addUser(String name) {\n        if (name.length() > 3) {\n            users.add(name);\n            db.save(name);\n            System.out.println(\"User added: \" + name);\n        } else {\n            System.out.println(\"Name too short\");\n        }\n    }\n\n    public void removeUser(String name) {\n        users.remove(name);\n        db.delete(name);\n        System.out.println(\"User removed: \" + name);\n    }\n\n    public List<String> getAllUsers() {\n        return users;\n    }\n\n    public void printUsers() {\n        for (String u : users) {\n            System.out.println(u);\n        }\n    }\n}\n\nclass DatabaseConnection {\n    public void save(String user

Corrected Java Code

In [15]:
import json

response_json = json.loads(clean_output)
java_content = response_json["output"]
print(java_content)

package app;

import java.util.ArrayList;
import java.util.List;

public class UserManager {
    private List<String> users = new ArrayList<>();
    private DatabaseConnection db = new DatabaseConnection();

    public void addUser(String name) {
        if (name.length() > 3) {
            users.add(name);
            db.save(name);
            System.out.println("User added: " + name);
        } else {
            System.out.println("Name too short");
        }
    }

    public void removeUser(String name) {
        users.remove(name);
        db.delete(name);
        System.out.println("User removed: " + name);
    }

    public List<String> getAllUsers() {
        return users;
    }

    public void printUsers() {
        for (String u : users) {
            System.out.println(u);
        }
    }
}

class DatabaseConnection {
    public void save(String user) {
        System.out.println("Saving user to database: " + user);
    }

    public void delete(String user) {
        Sys